In [ ]:
EXP_NAME = "3JUN_synth_3_1_2"
TRAINING_DATASET = "synth/3JUN/multivariate_biased_train.csv"
TEST_DATASET = "synth/3JUN/multivariate_biased_test.csv"
FEATURE_MAP = "scm/multivariate_synth_scm_config.json"
SEED = 4

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from notebooks.analysis_utils import mutual_info_grouped, plot_cat_feature, plot_cont_feature, plot_cf_cont_feature_comparison, plot_cf_cat_feature_comparison

from src.config import Config

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

In [ ]:
test_latents = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/test_latent_space.csv')
test_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/test_counterfactuals.csv')
training_latents = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/train_latent_space.csv')
training_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/train_counterfactuals.csv')
test_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TEST_DATASET}')
training_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TRAINING_DATASET}')

downstream_probe_results = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/downstream_probe_results.csv')

with open(f"{PROJECT_ROOT}/configs/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

In [ ]:
training_df = training_latents.merge(training_dataset, how="inner", left_on="patient_index", right_index=True)
training_df = training_df.merge(training_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
training_df.drop('sample_index_cf', axis=1, inplace=True)

test_df = test_latents.merge(test_dataset, how="inner", left_on="patient_index", right_index=True)
test_df = test_df.merge(test_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
test_df.drop('sample_index_cf', axis=1, inplace=True)

In [ ]:
desc_cols = [f['name'] for f in feature_map['desc']]
desc_cf_cols = [f"{f['name']}_cf" for f in feature_map['desc']]
latent_cols = [col for col in training_latents.columns if 'u_desc_' in col]
sens_col = feature_map['sens'][0]['name']
target_col = feature_map['target']['name']

# Downstream probe results

In [ ]:
columns = [c.removeprefix("u_") for c in downstream_probe_results.filter(regex="u_.*").columns]
formatted_results = pd.DataFrame(columns=["probe"], data=["x", "u"])
for col in columns:
  melted_subresults = downstream_probe_results.melt(value_vars=[f"x_{col}", f"u_{col}"], value_name=col, var_name="probe")
  melted_subresults['probe'] = melted_subresults['probe'].str[0]
  formatted_results = formatted_results.merge(melted_subresults, on="probe", how="outer")
formatted_results.set_index("probe", inplace=True)
formatted_results.sort_index(ascending=False, inplace=True)

def format_score(score):
  return round(score*100, 2)

print("---- GLOBAL PERFORMANCE ----\n")
print(formatted_results.filter(regex="global_.*").apply(format_score).to_markdown())
print("\n---- GROUP 0 ----\n")
print(formatted_results.filter(regex="0_.*").apply(format_score).to_markdown())
print("\n---- GROUP 1 ----\n")
print(formatted_results.filter(regex="1_.*").apply(format_score).to_markdown())

# Latent space

## Mutual Information with $X_{sens}$

### Training dataset

In [ ]:
mutual_info_grouped(
  training_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=400,
  seed=SEED
)

### Test dataset

In [ ]:
mutual_info_grouped(
  test_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=100,
  seed=SEED
)

# Counterfactuals

In [ ]:
# training_df.head()

In [ ]:
for feature in feature_map['desc']:
  col_name = feature['name']
  cf_col_name = feature['name'] + "_cf"
  group_0_mask = training_df[sens_col] == 0
  group_1_mask = training_df[sens_col] == 1
  if feature['type'] == "continuous":
    plot_cf_cont_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)
  else:
    plot_cf_cat_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)